# Building and integrating a testing framework

This notebook demonstrates how to use the testing framework to evaluate different LLM models.

## Models to Test
- gpt-5.2
- gpt-5-mini
- gpt-5-nano
- gpt-4.1-nano
- gpt-4o
- gpt-4o-mini
- gpt-4.1-mini

## Setup and Imports

In [1]:
import sys
import os
from datetime import datetime
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Add project root to path
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sql_ai_agent.testing import TestRunner, get_test_cases
from sql_ai_agent.testing.test_cases import get_test_summary

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## View Test Cases

In [2]:
# Get test summary
summary = get_test_summary()

print(f"Total Test Cases: {summary['total_tests']}")
print(f"\nBy Difficulty:")
for diff, count in sorted(summary['by_difficulty'].items()):
    print(f"  {diff.capitalize()}: {count}")

print(f"\nBy Category:")
for cat, count in sorted(summary['by_category'].items()):
    print(f"  {cat}: {count}")

Total Test Cases: 20

By Difficulty:
  Easy: 5
  Hard: 7
  Medium: 8

By Category:
  aggregation: 3
  basic: 2
  comparison: 1
  complex_filtering: 1
  edge_case: 3
  filtering: 2
  growth_analysis: 2
  market_share: 1
  percentage: 1
  ranking: 2
  time_series: 2


In [3]:
# Display all test cases
all_tests = get_test_cases()

test_df = pd.DataFrame([
    {
        'ID': test.id,
        'Difficulty': test.difficulty,
        'Category': test.category,
        'Question': test.question[:80] + '...' if len(test.question) > 80 else test.question
    }
    for test in all_tests
])

display(test_df)

,ID,Difficulty,Category,Question
0,1,easy,basic,How many total rows are in the air_traffic table?
1,2,easy,aggregation,What are the top 5 airlines by total passenger count in 2024?
2,3,easy,aggregation,"Show total passengers by terminal, excluding transit passengers"
3,4,easy,filtering,How many international passengers arrived at SFO in 2023?
4,5,easy,basic,List all distinct operating airlines in the database
5,6,medium,comparison,Compare domestic vs international passenger traffic for 2024
6,7,medium,aggregation,What are the top 3 low-fare carriers by passenger count in 2024?
7,8,medium,time_series,Show monthly passenger trends for United Airlines in 2024
8,9,medium,ranking,Which terminal had the highest passenger traffic in January 2024?
9,10,medium,percentage,What percentage of total passengers were on low-fare carriers in 2024?


## Initialize Test Runner

In [4]:
# Initialize the test runner with database connection
runner = TestRunner(
    db_host="localhost",
    db_port=5432,
    db_name="my_db",
    db_user="postgres",
    db_password="password",
    table_name="air_traffic"
)

print("✅ Test runner initialized successfully")

✅ Test runner initialized successfully


## Define Models to Test

In [5]:
# Define models to test
# models_to_test = {
#     'openai': [
#         'gpt-5.2',
#         'gpt-5-mini',
#         'gpt-5-nano',
#         'gpt-4.1-nano',
#         'gpt-4o',
#         'gpt-4o-mini',
#         'gpt-4.1-mini'
#     ]
# }

models_to_test = {
    "openai": [
        "gpt-4o-mini",
        "gpt-4o",
        "gpt-5-mini",
        "gpt-5-nano",
    ]
}

print("Models to test:")
for provider, models in models_to_test.items():
    print(f"\n{provider.upper()}:")
    for model in models:
        print(f"  - {model}")

Models to test:

OPENAI:
  - gpt-4o-mini
  - gpt-4o
  - gpt-5-mini
  - gpt-5-nano


## Option 1: Run All Tests for All Models

This will run all 20 test cases + 5 debug tests for each model.

In [6]:
# Run all tests for all models
# WARNING: This may take a while (20 tests + 5 debug tests per model)

print("Starting comprehensive test run...\n")
print("This will test:")
print(f"  - {len(models_to_test['openai'])} models")
print(f"  - 10 query generation tests per model")
print(f"  - 5 debug mechanism tests per model")
print(f"  - Total: {len(models_to_test['openai']) * 10} test executions\n")

results_df = runner.run_all_tests(
    providers=['openai'],
    models=models_to_test,
    max_debug_trials=3
)

print("\n✅ All tests completed!")
print(f"Total results: {len(results_df)} test executions")

Starting comprehensive test run...

This will test:
  - 4 models
  - 10 query generation tests per model
  - 5 debug mechanism tests per model
  - Total: 40 test executions


################################################################################
# Testing Provider: OPENAI | Model: gpt-4o-mini
################################################################################

Testing openai/gpt-4o-mini - Query Generation (20 tests)

[1/20] Test 1: How many total rows are in the air_traffic table?...
   ✓ PASS (0.66s) - Query executed successfully
[2/20] Test 2: What are the top 5 airlines by total passenger count in 2024...
   ✗ FAIL (2.03s) - Custom validation failed
[3/20] Test 3: Show total passengers by terminal, excluding transit passeng...
   ✓ PASS (1.03s) - Query executed successfully
[4/20] Test 4: How many international passengers arrived at SFO in 2023?...
   ✓ PASS (0.92s) - Query executed successfully
[5/20] Test 5: List all distinct operating airlines in the databa

Argument 'format' is not supported for expression 'ToChar' when targeting Dialect.


   ✓ PASS (6.01s) - Query executed successfully
[9/20] Test 9: Which terminal had the highest passenger traffic in January ...
   ✓ PASS (4.92s) - Query executed successfully
[10/20] Test 10: What percentage of total passengers were on low-fare carrier...
   ✓ PASS (8.18s) - Query executed successfully
[11/20] Test 11: Show passenger count by boarding area for Terminal 1 in 2024...
   ✓ PASS (3.78s) - Query executed successfully
[12/20] Test 12: Which GEO Region had the most international passengers in 20...
   ✗ FAIL (8.21s) - Too few rows: 0 < 1
[13/20] Test 13: Calculate year-over-year growth percentage for total passeng...
   ✓ PASS (6.97s) - Query executed successfully
[14/20] Test 14: Which 5 airlines had the highest year-over-year growth from ...
   ✓ PASS (8.87s) - Query executed successfully
[15/20] Test 15: Show quarterly passenger trends for 2024, broken down by dom...
   ✓ PASS (17.73s) - Query executed successfully
[16/20] Test 16: What is the market share percentage of ea

Argument 'format' is not supported for expression 'ToChar' when targeting Dialect.


   ✓ PASS (7.35s) - Query executed successfully
[19/20] Test 19: Compare passenger counts using Operating Airline vs Publishe...
   ✓ PASS (18.36s) - Query executed successfully
[20/20] Test 20: What is the average passenger count per flight by terminal i...
   ✓ PASS (6.62s) - Query executed successfully

Testing openai/gpt-5-mini - Debug Mechanism (5 tests)



...

[1/5] Debug Test 1: Missing quotes around column names with spaces
   ✓ FIXED in 1 trial(s) (3.44s)
[2/5] Debug Test 2: Wrong table name (missing underscore)
   ✓ FIXED in 1 trial(s) (3.31s)
[3/5] Debug Test 3: Missing GROUP BY for aggregation
   ✓ FIXED in 1 trial(s) (2.22s)
[4/5] Debug Test 4: Wrong column name (misspelled)
   ✓ FIXED in 1 trial(s) (2.83s)
[5/5] Debug Test 5: Invalid date comparison syntax
   ✓ FIXED in 1 trial(s) (3.97s)

################################################################################
# Testing Provider: OPENAI | Model: gpt-5-nano
################################################################################

Testing openai/gpt-5-nano - Query Generation (20 tests)

[1/20] Test 1: How many total rows are in the air_traffic table?...
   ✓ PASS (2.60s) - Query executed successfully
[2/20] Test 2: What are the top 5 airlines by total passenger count in 2024...
   ✗ FAIL (10.05s) - Custom validation failed
[3/20] Test 3: Show total passengers by termin

Argument 'format' is not supported for expression 'ToChar' when targeting Dialect.


   ✓ PASS (20.38s) - Query executed successfully
[16/20] Test 16: What is the market share percentage of each airline in 2024?...
   ✓ PASS (14.28s) - Query executed successfully
[17/20] Test 17: Find airlines that operate in both terminals and have more t...
   ✓ PASS (6.49s) - Query executed successfully
[18/20] Test 18: Show total enplaned passengers only (departures) for each mo...


Argument 'format' is not supported for expression 'ToChar' when targeting Dialect.


   ✓ PASS (21.14s) - Query executed successfully
[19/20] Test 19: Compare passenger counts using Operating Airline vs Publishe...
   ✓ PASS (16.98s) - Query executed successfully
[20/20] Test 20: What is the average passenger count per flight by terminal i...
   ✓ PASS (10.13s) - Query executed successfully

Testing openai/gpt-5-nano - Debug Mechanism (5 tests)

[1/5] Debug Test 1: Missing quotes around column names with spaces
   ✓ FIXED in 2 trial(s) (15.40s)
[2/5] Debug Test 2: Wrong table name (missing underscore)
   ✓ FIXED in 1 trial(s) (3.67s)
[3/5] Debug Test 3: Missing GROUP BY for aggregation
   ✓ FIXED in 1 trial(s) (3.19s)
[4/5] Debug Test 4: Wrong column name (misspelled)
   ✓ FIXED in 1 trial(s) (4.55s)
[5/5] Debug Test 5: Invalid date comparison syntax
   ✓ FIXED in 1 trial(s) (5.27s)

✅ All tests completed!
Total results: 100 test executions


## Save Results to CSV

In [7]:
# Generate timestamp for file naming
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Save detailed results
results_file = f'test_results_{timestamp}.csv'
results_df.to_csv(results_file, index=False)
print(f"✅ Detailed results saved to: {results_file}")

# Generate and save summary
summary_df = runner.generate_summary_report(results_df)
summary_file = f'test_summary_{timestamp}.csv'
summary_df.to_csv(summary_file, index=False)
print(f"✅ Summary report saved to: {summary_file}")

✅ Detailed results saved to: test_results_20260524_000402.csv
✅ Summary report saved to: test_summary_20260524_000402.csv


## View Summary Results

In [8]:
# Display summary report
print("\n" + "="*100)
print("TEST SUMMARY")
print("="*100 + "\n")

display(summary_df)


TEST SUMMARY



,provider,model,test_type,total_tests,successful,failed,success_rate,avg_execution_time,avg_trials_to_fix
0,openai,gpt-4o,debug_mechanism,5,5,0,100.00%,0.610s,1.00
1,openai,gpt-4o,query_generation,20,16,4,80.00%,0.827s,NaN
2,openai,gpt-4o-mini,debug_mechanism,5,5,0,100.00%,0.967s,1.20
3,openai,gpt-4o-mini,query_generation,20,14,6,70.00%,1.622s,NaN
4,openai,gpt-5-mini,debug_mechanism,5,5,0,100.00%,3.155s,1.00
5,openai,gpt-5-mini,query_generation,20,17,3,85.00%,7.886s,NaN
6,openai,gpt-5-nano,debug_mechanism,5,5,0,100.00%,6.415s,1.20
7,openai,gpt-5-nano,query_generation,20,16,4,80.00%,12.276s,NaN


## Analyze Results by Test Type

In [9]:
# Query Generation Results
query_results = results_df[results_df['test_type'] == 'query_generation'].copy()

print("\n" + "="*80)
print("QUERY GENERATION RESULTS")
print("="*80 + "\n")

query_summary = query_results.groupby('model').agg({
    'success': ['count', 'sum', 'mean'],
    'execution_time': 'mean'
}).round(3)

query_summary.columns = ['Total Tests', 'Successful', 'Success Rate', 'Avg Time (s)']
query_summary['Success Rate'] = (query_summary['Success Rate'] * 100).round(2).astype(str) + '%'

display(query_summary.sort_values('Successful', ascending=False))


QUERY GENERATION RESULTS



,Total Tests,Successful,Success Rate,Avg Time (s)
model,,,,
gpt-5-mini,20,17,85.0%,7.886
gpt-4o,20,16,80.0%,0.827
gpt-5-nano,20,16,80.0%,12.276
gpt-4o-mini,20,14,70.0%,1.622


In [10]:
# Debug Mechanism Results
debug_results = results_df[results_df['test_type'] == 'debug_mechanism'].copy()

if not debug_results.empty:
    print("\n" + "="*80)
    print("DEBUG MECHANISM RESULTS")
    print("="*80 + "\n")
    
    debug_summary = debug_results.groupby('model').agg({
        'success': ['count', 'sum', 'mean'],
        'trials_needed': 'mean',
        'execution_time': 'mean'
    }).round(3)
    
    debug_summary.columns = ['Total Tests', 'Fixed', 'Fix Rate', 'Avg Trials', 'Avg Time (s)']
    debug_summary['Fix Rate'] = (debug_summary['Fix Rate'] * 100).round(2).astype(str) + '%'
    
    display(debug_summary.sort_values('Fixed', ascending=False))


DEBUG MECHANISM RESULTS



,Total Tests,Fixed,Fix Rate,Avg Trials,Avg Time (s)
model,,,,,
gpt-4o,5,5,100.0%,1.0,0.610
gpt-4o-mini,5,5,100.0%,1.2,0.967
gpt-5-mini,5,5,100.0%,1.0,3.155
gpt-5-nano,5,5,100.0%,1.2,6.415


## Analyze Results by Difficulty

In [11]:
# Success rate by difficulty level
if 'difficulty' in query_results.columns:
    print("\n" + "="*80)
    print("SUCCESS RATE BY DIFFICULTY")
    print("="*80 + "\n")
    
    difficulty_pivot = pd.crosstab(
        query_results['model'],
        query_results['difficulty'],
        query_results['success'],
        aggfunc='mean'
    ).round(3) * 100
    
    # Reorder columns: easy, medium, hard
    column_order = [col for col in ['easy', 'medium', 'hard'] if col in difficulty_pivot.columns]
    difficulty_pivot = difficulty_pivot[column_order]
    
    display(difficulty_pivot.style.format("{:.2f}%"))


SUCCESS RATE BY DIFFICULTY



difficulty,easy,medium,hard
model,,,
gpt-4o,80.00%,62.50%,100.00%
gpt-4o-mini,80.00%,50.00%,85.70%
gpt-5-mini,80.00%,75.00%,100.00%
gpt-5-nano,80.00%,62.50%,100.00%


## Visualizations

In [12]:
# Success Rate Comparison
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Prepare data
query_success = query_results.groupby('model')['success'].mean().sort_values(ascending=False) * 100
query_success_df = query_success.reset_index()
query_success_df.columns = ['model', 'success_rate']

# Create subplots with 1 row and 2 columns
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Query Generation Success Rate by Model', 'Debug Fix Rate by Model')
)

# Query Generation Success Rate (left subplot)
fig.add_trace(
    go.Bar(
        x=query_success_df['model'],
        y=query_success_df['success_rate'],
        name='Query Success',
        marker_color='steelblue',
        showlegend=False
    ),
    row=1, col=1
)

# Debug Fix Rate (right subplot)
if not debug_results.empty:
    debug_success = debug_results.groupby('model')['success'].mean().sort_values(ascending=False) * 100
    debug_success_df = debug_success.reset_index()
    debug_success_df.columns = ['model', 'fix_rate']
    
    fig.add_trace(
        go.Bar(
            x=debug_success_df['model'],
            y=debug_success_df['fix_rate'],
            name='Debug Fix',
            marker_color='coral',
            showlegend=False
        ),
        row=1, col=2
    )

# Update layout
fig.update_xaxes(title_text="Model", row=1, col=1)
fig.update_xaxes(title_text="Model", row=1, col=2)
fig.update_yaxes(title_text="Success Rate (%)", range=[0, 100], row=1, col=1)
fig.update_yaxes(title_text="Fix Rate (%)", range=[0, 100], row=1, col=2)

fig.update_layout(
    height=500,
    width=1200,
    showlegend=False,
    template='plotly_white'
)

fig.write_html(f'success_rates_{timestamp}.html')
fig.show()

In [13]:
# Execution Time Comparison
import plotly.graph_objects as go

avg_times = query_results.groupby('model')['execution_time'].mean().sort_values()
avg_times_df = avg_times.reset_index()
avg_times_df.columns = ['model', 'avg_time']

fig = go.Figure()

fig.add_trace(go.Bar(
    x=avg_times_df['avg_time'],
    y=avg_times_df['model'],
    orientation='h',
    marker_color='seagreen',
    text=avg_times_df['avg_time'].round(3),
    textposition='auto',
))

fig.update_layout(
    title='Average Query Execution Time by Model',
    xaxis_title='Average Execution Time (seconds)',
    yaxis_title='Model',
    height=500,
    width=900,
    template='plotly_white',
    showlegend=False
)

fig.write_html(f'execution_times_{timestamp}.html')
fig.show()

In [14]:
# Success Rate by Difficulty (Heatmap)
if 'difficulty' in query_results.columns:
    import plotly.graph_objects as go
    
    difficulty_pivot = pd.crosstab(
        query_results['model'],
        query_results['difficulty'],
        query_results['success'],
        aggfunc='mean'
    ) * 100
    
    # Reorder columns: easy, medium, hard
    column_order = [col for col in ['easy', 'medium', 'hard'] if col in difficulty_pivot.columns]
    difficulty_pivot = difficulty_pivot[column_order]
    
    # Create heatmap
    fig = go.Figure(data=go.Heatmap(
        z=difficulty_pivot.values,
        x=difficulty_pivot.columns,
        y=difficulty_pivot.index,
        colorscale='RdYlGn',
        zmin=0,
        zmax=100,
        text=difficulty_pivot.values.round(1),
        texttemplate='%{text:.1f}',
        textfont={"size": 12},
        colorbar=dict(title="Success Rate (%)")
    ))
    
    fig.update_layout(
        title='Success Rate by Model and Difficulty',
        xaxis_title='Difficulty Level',
        yaxis_title='Model',
        height=600,
        width=800,
        template='plotly_white'
    )
    
    fig.write_html(f'difficulty_heatmap_{timestamp}.html')
    fig.show()

## Detailed Failure Analysis

In [15]:
# Show failed query generation tests
failed_queries = query_results[query_results['success'] == False].copy()

if not failed_queries.empty:
    print("\n" + "="*80)
    print(f"FAILED QUERY TESTS ({len(failed_queries)} failures)")
    print("="*80 + "\n")
    
    failure_summary = failed_queries.groupby(['model', 'test_id']).size().reset_index(name='count')
    
    print("Failures by Model and Test ID:")
    display(failure_summary.pivot(index='test_id', columns='model', values='count').fillna(0).astype(int))
    
    # Show details of failed tests
    print("\nFailed Test Details:")
    failed_details = failed_queries[[
        'model', 'test_id', 'question', 'difficulty', 'category', 
        'error_message', 'validation_message'
    ]].head(10)
    display(failed_details)
else:
    print("\n🎉 All query generation tests passed!")


FAILED QUERY TESTS (17 failures)

Failures by Model and Test ID:


model,gpt-4o,gpt-4o-mini,gpt-5-mini,gpt-5-nano
test_id,,,,
2,1,1,1,1
6,1,1,1,1
7,1,0,0,1
11,0,1,0,0
12,1,1,1,1
17,0,1,0,0
18,0,1,0,0



Failed Test Details:


,model,test_id,question,difficulty,category,error_message,validation_message
1,gpt-4o-mini,2,What are the top 5 airlines by total passenger count in 2024?,easy,aggregation,NaN,Custom validation failed
5,gpt-4o-mini,6,Compare domestic vs international passenger traffic for 2024,medium,comparison,NaN,Custom validation failed
10,gpt-4o-mini,11,Show passenger count by boarding area for Terminal 1 in 2024,medium,filtering,NaN,Too few rows: 0 < 1
11,gpt-4o-mini,12,Which GEO Region had the most international passengers in 2023?,medium,ranking,NaN,Too few rows: 0 < 1
16,gpt-4o-mini,17,Find airlines that operate in both terminals and have more than 100000 passengers in 2024,hard,complex_filtering,NaN,Custom validation failed
17,gpt-4o-mini,18,Show total enplaned passengers only (departures) for each month in 2024,medium,edge_case,NaN,Too few rows: 0 < 1
26,gpt-4o,2,What are the top 5 airlines by total passenger count in 2024?,easy,aggregation,NaN,Custom validation failed
30,gpt-4o,6,Compare domestic vs international passenger traffic for 2024,medium,comparison,NaN,Custom validation failed
31,gpt-4o,7,What are the top 3 low-fare carriers by passenger count in 2024?,medium,aggregation,NaN,Too few rows: 0 < 1
36,gpt-4o,12,Which GEO Region had the most international passengers in 2023?,medium,ranking,NaN,Too few rows: 0 < 1


In [16]:
# Show failed debug tests
if not debug_results.empty:
    failed_debug = debug_results[debug_results['success'] == False].copy()
    
    if not failed_debug.empty:
        print("\n" + "="*80)
        print(f"FAILED DEBUG TESTS ({len(failed_debug)} failures)")
        print("="*80 + "\n")
        
        failed_debug_details = failed_debug[[
            'model', 'test_id', 'description', 'error_type', 
            'trials_needed', 'final_error'
        ]]
        display(failed_debug_details)
    else:
        print("\n🎉 All debug tests passed!")


🎉 All debug tests passed!


## Model Comparison Summary

In [17]:
# Create comprehensive comparison table
comparison_data = []

for model in models_to_test['openai']:
    model_query = query_results[query_results['model'] == model]
    model_debug = debug_results[debug_results['model'] == model] if not debug_results.empty else pd.DataFrame()
    
    row = {
        'Model': model,
        'Query Tests': len(model_query),
        'Query Success': model_query['success'].sum(),
        'Query Success %': f"{(model_query['success'].mean() * 100):.2f}%" if len(model_query) > 0 else 'N/A',
        'Avg Query Time (s)': f"{model_query['execution_time'].mean():.3f}" if len(model_query) > 0 else 'N/A',
        'Debug Tests': len(model_debug),
        'Debug Fixed': model_debug['success'].sum() if len(model_debug) > 0 else 0,
        'Debug Fix %': f"{(model_debug['success'].mean() * 100):.2f}%" if len(model_debug) > 0 else 'N/A',
        'Avg Trials': f"{model_debug[model_debug['success'] == True]['trials_needed'].mean():.2f}" if len(model_debug[model_debug['success'] == True]) > 0 else 'N/A'
    }
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*120)
print("COMPREHENSIVE MODEL COMPARISON")
print("="*120 + "\n")

display(comparison_df)

# Save comparison
comparison_df.to_csv(f'model_comparison_{timestamp}.csv', index=False)
print(f"\n✅ Model comparison saved to: model_comparison_{timestamp}.csv")


COMPREHENSIVE MODEL COMPARISON



,Model,Query Tests,Query Success,Query Success %,Avg Query Time (s),Debug Tests,Debug Fixed,Debug Fix %,Avg Trials
0,gpt-4o-mini,20,14,70.00%,1.622,5,5,100.00%,1.20
1,gpt-4o,20,16,80.00%,0.827,5,5,100.00%,1.00
2,gpt-5-mini,20,17,85.00%,7.886,5,5,100.00%,1.00
3,gpt-5-nano,20,16,80.00%,12.276,5,5,100.00%,1.20



✅ Model comparison saved to: model_comparison_20260524_000402.csv


## Key Findings Summary

In [18]:
print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80 + "\n")

# Best performing model for query generation
best_query_model = query_results.groupby('model')['success'].mean().idxmax()
best_query_rate = query_results.groupby('model')['success'].mean().max() * 100
print(f"🏆 Best Query Generation Model: {best_query_model} ({best_query_rate:.2f}% success rate)")

# Fastest model
fastest_model = query_results.groupby('model')['execution_time'].mean().idxmin()
fastest_time = query_results.groupby('model')['execution_time'].mean().min()
print(f"⚡ Fastest Model: {fastest_model} ({fastest_time:.3f}s avg execution time)")

# Best debug model
if not debug_results.empty:
    best_debug_model = debug_results.groupby('model')['success'].mean().idxmax()
    best_debug_rate = debug_results.groupby('model')['success'].mean().max() * 100
    print(f"🔧 Best Debug Model: {best_debug_model} ({best_debug_rate:.2f}% fix rate)")

# Overall statistics
total_tests = len(results_df)
total_success = results_df['success'].sum()
overall_rate = (total_success / total_tests * 100) if total_tests > 0 else 0
print(f"\n📊 Overall Statistics:")
print(f"   Total Tests: {total_tests}")
print(f"   Total Successful: {total_success}")
print(f"   Overall Success Rate: {overall_rate:.2f}%")

print("\n" + "="*80)


KEY FINDINGS

🏆 Best Query Generation Model: gpt-5-mini (85.00% success rate)
⚡ Fastest Model: gpt-4o (0.827s avg execution time)
🔧 Best Debug Model: gpt-4o (100.00% fix rate)

📊 Overall Statistics:
   Total Tests: 100
   Total Successful: 83
   Overall Success Rate: 83.00%

